# LigandMPNN Inverse Folding

This notebook demonstrates how to use `run_ligandmpnn_sample` to design protein sequences conditioned on both backbone structure and nearby ligand atoms. LigandMPNN extends ProteinMPNN by incorporating small molecule geometry into sequence design, enabling the creation of binding pocket sequences that are compatible with a specific ligand conformation. When no ligand atoms are present in the input structure, LigandMPNN behaves similarly to ProteinMPNN.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ligandmpnn")
display_overview("ligandmpnn")
display_docs_section("ligandmpnn", "Background")

# LigandMPNN

> LigandMPNN is an inverse folding model that designs protein sequences conditioned on a backbone structure and its molecular context -- including bound ligands, metal ions, nucleic acids, and non-standard residues. It extends ProteinMPNN's message-passing neural network architecture to incorporate atomic-level information from the non-protein environment surrounding the design target.

## Background

[Inverse folding](https://en.wikipedia.org/wiki/Protein_design#Inverse_folding) solves the "reverse" protein design problem: given a desired 3D backbone structure, what amino acid sequence will fold into that structure? This is the complement of structure prediction (sequence -> structure).

ProteinMPNN (Dauparas et al., 2022) pioneered the [message-passing neural network](https://en.wikipedia.org/wiki/Graph_neural_network) approach for inverse folding, achieving recovery rates (\~50% native sequence identity) far exceeding previous physics-based methods. However, ProteinMPNN only considers the protein backbone and ignores non-protein molecules.

LigandMPNN extends this by encoding the atomic coordinates and chemical identities of:

* **Small-molecule ligands** (drugs, metabolites, [cofactors](https://en.wikipedia.org/wiki/Cofactor_\(biochemistry\)))
* **Metal ions** (Zn, Fe, Mg, Ca, etc.)
* **Nucleic acids** (DNA and RNA)
* **Non-standard residues** (modified amino acids, [post-translational modifications](https://en.wikipedia.org/wiki/Post-translational_modification))

This context is critical for designing functional enzymes, metalloprotein binding sites, and protein-nucleic acid interfaces, where the sequence must be compatible with both the protein fold and its molecular partners.

## Available tools

In [2]:
display_available_tools("ligandmpnn")

- **`run_ligandmpnn_sample()`** — Sample protein sequences using LigandMPNN

### `run_ligandmpnn_sample`

LigandMPNN analyzes the backbone coordinates and any ligand atoms present in the structure to generate sequences optimized for the target fold and binding environment. We use GFP as a minimal runnable example. For ligand-aware design, provide a structure file that includes HETATM records for the bound ligand of interest. You can optionally fix residue positions to preserve critical interactions and exclude specific amino acids from the designed sequences.

In [3]:
from proto_tools import (
    InverseFoldingStructureInput,
    LigandMPNNSampleInput,
    LigandMPNNSampleConfig,
    run_ligandmpnn_sample,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_sample")

# Load GFP structure and configure design constraints
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    chains_to_redesign=["A"],
    fixed_positions={"A": [2, 3, 4]},
)
inputs = LigandMPNNSampleInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[InverseFoldingStructureInput]` | required | List of InverseFoldingInput objects, each containing a structure and optional chain\_ids/fixed\_positions constraints. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. Automatically converted to Structure. |
| `chain_ids` | `array` |  | Optional list of chain IDs to design. If None, all chains in the structure are designed. |
| `fixed_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions to keep fixed during design. Positions are 1-indexed. |

In [5]:
# Display config docs
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_sample")

# Configure sampling | see docs above for all fields
config = LigandMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs
    excluded_amino_acids=["C"],     # Exclude cysteine
    seed=42,
    device="cuda",                  # Change to "cpu" if no GPU available
)

**Config** — `InverseFoldingConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `num_sequences_per_structure` | `integer` | `1` | Total number of sequences to generate for each input structure. Sequences are generated in batches of `batch_size`. Defaults to 1. |
| `batch_size` | `integer` |  | Number of sequences to process simultaneously on GPU. Larger batches improve throughput but use more GPU memory; reduce if encountering out-of-memory errors. Defaults to `num_sequences_per_structure`. |
| `temperature` | `number` | `0.1` | Controls randomness in sampling from logits. Defaults to 0.1. |
| `excluded_amino_acids` | `array` |  | List of amino acids that are not allowed in the sequence. If None, no amino acids will be disallowed. C is commonly specified. Defaults to None. |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution), or specific GPU devices like 'cuda:0'. Defaults to 'cuda'. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
result = run_ligandmpnn_sample(inputs, config)

LigandMPNN sampling:   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_sample")

# Print designed sequences and compare to the original GFP sequence
chain_a = gfp_structure.get_chain_sequence("A", remove_non_standard=True)

for i, seq_res in enumerate(result.designed_sequences):
    print(f"Structure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        preview = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"  Design {j}: {preview}")
        print(f"            Length: {len(seq)} residues")

**Output** — `InverseFoldingOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_sequences` | `List[DesignedSequences]` | required | List of DesignedSequences objects, one for each input structure. The order matches the input structures order. |
| `sequences` | `List[string]` | required | Designed amino acid sequences in single-letter code. |

Structure 0 designs:
  Design 1: SKGAELMKGEVPVKVTLEGDVNGKKFKVEGEGHVRAQEGLLELHFECTSG...
            Length: 227 residues
  Design 2: SKGAELLKGRVPVRVHLEGDVNGKKFKVEGEGHGDASRGLLRLHFRCTSG...
            Length: 227 residues


## Export Results

Designed sequences can be saved to standard file formats for downstream analysis. JSON format is convenient for preserving the full LigandMPNN metrics alongside the sequences, while FASTA format is useful for feeding results into other sequence analysis tools.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="ligandmpnn_designs", export_path=output_dir, file_format="json")
print(f"Results exported to {output_dir / 'ligandmpnn_designs/'}")

Results exported to example_output/ligandmpnn_designs
